# RoomGuard — UAS Workshop Machine Learning for IoT (Kelompok 12)
**Judul:** RoomGuard: Sistem Web Klasifikasi Keberadaan Orang dalam Ruangan Menggunakan Decision Tree Berbasis Sensor Ambience Smartphone

**Kelompok 12:** Evodia Maribet Kristina Manalu & Hasbi

Sistem web yang mengklasifikasikan status keberadaan orang (Terisi / Kosong) dari sensor cahaya (Lux) dan suara (dB) smartphone. Dataset: 2.000 baris — 1.500 publik (UCI Room Occupancy Estimation, ID 864) + 500 private (phyphox, interval 30 detik).

Dataset publik: https://archive.ics.uci.edu/dataset/864/room+occupancy+estimation

Jalankan sel dari atas ke bawah (Runtime > Run all).

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
print('pandas', pd.__version__)

## 1. Download dataset publik UCI 864 (10.129 baris, file asli tanpa diubah)

In [ ]:
!curl -L -o uci864.zip "https://archive.ics.uci.edu/static/public/864/room+occupancy+estimation.zip"
!unzip -o -q uci864.zip -d uci864 && ls -lh uci864/
uci = pd.read_csv('uci864/Occupancy_Estimation.csv')
print(uci.shape)
print(uci['Room_Occupancy_Count'].value_counts().sort_index())
uci.head()

## 2. Bangun fitur — hanya Light (Lux) dan Sound (dB), sesuai penugasan
`Light_Lux` = rata-rata S1–S4 Light (satuannya sudah Lux, sama seperti phyphox). Suara UCI tersimpan dalam **Volt** (keluaran amplifier mikrofon), jadi dikonversi ke dB agar satuannya sama dengan phyphox: `Sound_dB = 20*log10(V/0,05) + 35` (V=0,05 → 35 dB, V=0,30 → ±50,6 dB). Target biner: `Occupied = 1` jika `Room_Occupancy_Count > 0`.

In [ ]:
uci['uci_row_id'] = uci.index
uci['Occupied'] = (uci['Room_Occupancy_Count'] > 0).astype(int)

def sound_v_to_db(v):
    return 20*np.log10(np.maximum(v, 0.01)/0.05) + 35

pub = (uci.groupby('Occupied', group_keys=False)
          .apply(lambda g: g.sample(n=int(round(1500*len(g)/len(uci))), random_state=42)))
pub = pub.sample(frac=1, random_state=42).reset_index(drop=True)
pub['Light_Lux'] = pub[['S1_Light','S2_Light','S3_Light','S4_Light']].mean(axis=1).round(2)
pub['Sound_dB'] = sound_v_to_db(pub[['S1_Sound','S2_Sound','S3_Sound','S4_Sound']].mean(axis=1)).round(2)
pub['Label'] = pub['Occupied'].map({0:'Empty', 1:'Occupied'})
print('public sample:', pub.shape, '| Empty:', (pub.Occupied==0).sum(), '| Occupied:', (pub.Occupied==1).sum())
print(pub.groupby('Occupied')[['Light_Lux','Sound_dB']].mean().round(2))

## 3. Data private (phyphox) — 500 baris
File `private_phyphox_500.csv` di repo ini dibangkitkan mengikuti protokol phyphox (seed tetap) agar sistem bisa dilatih sekarang — **bukan rekaman asli**. Sebelum presentasi, ganti dengan rekaman phyphox asli: upload file CSV asli (`timestamp, Light_Lux, Sound_dB, Occupied`) lewat sel di bawah, lalu lanjutkan. Pelabelan mengikuti standar tetap di `dataset/LABELING_STANDARD.md`: label dari lembar kehadiran yang ditulis **sebelum** nilai sensor dibaca, bukan dari nilai sensor.

In [ ]:
# Jika punya rekaman phyphox ASLI, jalankan sel ini dan upload file-nya (jika tidak, lewati).
# from google.colab import files
# uploaded = files.upload()  # pilih file private_phyphox_REAL.csv
# priv = pd.read_csv('private_phyphox_REAL.csv')
# print(priv.shape); print(priv.head())

# Stand-in terkalibrasi (dipakai bila belum ada rekaman asli):
rng = np.random.default_rng(7)
rows = []
for occ in [0, 1]:
    for i in range(250):
        u = rng.random()
        if occ == 0:
            light = rng.normal(4,4) if u<0.70 else (rng.normal(32,12) if u<0.90 else rng.normal(55,10))
            sound = rng.normal(36,2.5) if u<0.70 else (rng.normal(39,3) if u<0.90 else rng.normal(44,4))
        else:
            light = rng.normal(105,35) if u<0.70 else (rng.normal(38,12) if u<0.90 else rng.normal(10,6))
            sound = rng.normal(54,6) if u<0.70 else (rng.normal(47,5) if u<0.90 else rng.normal(40,3))
        rows.append({'Light_Lux': round(float(np.clip(light,0,220)),2), 'Sound_dB': round(float(np.clip(sound,30,75)),2), 'Occupied': occ})
priv = pd.DataFrame(rows)
df = pd.concat([pub[['Light_Lux','Sound_dB','Occupied']], priv], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print('combined:', df.shape, '| Empty:', (df.Occupied==0).sum(), '| Occupied:', (df.Occupied==1).sum())
df.to_csv('combined_dataset_2000.csv', index=False)

## 4. Split 70% latih / 30% uji (stratified) dan latih Decision Tree
Pemilihan model: grid search + 5-fold cross-validation, skor F1-macro (karena kelas tidak seimbang: Empty 73,4%). `min_samples_leaf >= 5` supaya tidak ada daun yang diputuskan oleh 1–4 sampel saja.

In [ ]:
X, y = df[['Light_Lux','Sound_dB']], df['Occupied']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
print('train:', len(X_train), '| test:', len(X_test))
param_grid = {'max_depth': [3,4,5,6,7,8,None], 'min_samples_leaf': [5,10], 'criterion': ['gini','entropy']}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(DecisionTreeClassifier(random_state=42, class_weight='balanced'), param_grid, cv=cv, scoring='f1_macro')
grid.fit(X_train, y_train)
model = grid.best_estimator_
print('best params:', grid.best_params_, '| CV macro-F1:', round(grid.best_score_,4))

## 5. Evaluasi pada data uji (600 baris yang belum pernah dilihat model)

In [ ]:
y_pred = model.predict(X_test)
print('Accuracy:', round(accuracy_score(y_test, y_pred),4))
print('Macro precision:', round(precision_score(y_test, y_pred, average='macro'),4))
print('Macro recall:', round(recall_score(y_test, y_pred, average='macro'),4))
print('Macro F1:', round(f1_score(y_test, y_pred, average='macro'),4))
print(classification_report(y_test, y_pred, target_names=['Empty','Occupied'], digits=4))
cm = confusion_matrix(y_test, y_pred)
print('Confusion matrix [[TN,FP],[FN,TP]]:'); print(cm)
fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Empty','Occupied']).plot(ax=ax, cmap='Blues')
ax.set_title('Confusion Matrix — Test Set (n=600)'); plt.show()
print('Feature importances:', dict(zip(['Light_Lux','Sound_dB'], model.feature_importances_.round(4))))
print(export_text(model, feature_names=['Light_Lux','Sound_dB'], class_names=['Empty','Occupied'])[:800])

## 6. Coba prediksi — 4 contoh (termasuk 2 hard case)

In [ ]:
def predict(light, sound):
    p = model.predict_proba([[light, sound]])[0]
    label = 'Occupied' if p[1] >= p[0] else 'Empty'
    print(f'Light {light} Lux, Sound {sound} dB -> {label} (Empty {p[0]*100:.1f}% / Occupied {p[1]*100:.1f}%)')
    return label
predict(120, 56)  # Occupied, terang + berisik
predict(5, 36)    # Empty, gelap + sepi
predict(90, 37)   # hard case: lampu menyala saat kosong -> salah diklasifikasi Occupied (95,4%)
predict(8, 52)    # hard case: gelap tapi berisik -> Occupied

## 7. Simpan model (untuk aplikasi web Flask di `app/app.py`)

In [ ]:
import joblib, json
joblib.dump(model, 'decision_tree_model.pkl')
print('model tersimpan: decision_tree_model.pkl')
print('Jalankan aplikasi web di laptop: pip install -r requirements.txt && python app/app.py -> http://127.0.0.1:5000')